## Import Libraries

In [ ]:
from pprint import pprint

import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchmetrics.classification import Accuracy, Precision, Recall
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.datasets import ImageFolder


## Datasets

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_dir = "../data"
CrackDataset = ImageFolder(root=img_dir)

train_size = int(0.7 * len(CrackDataset))
test_size = int(0.15 * len(CrackDataset))
val_size = len(CrackDataset) - train_size - test_size

train_dataset, test_dataset, val_dataset = random_split(CrackDataset, [train_size, test_size, val_size])

train_dataset.transforms = train_transforms
test_dataset.transforms = val_transforms
val_dataset.transforms = val_transforms


## Flexible Neural Network

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, num_filters):
        super(ConvBlock, self).__init__()

        self.num_filters = num_filters

        self.conv = nn.Conv2d(in_channels = num_filters, out_channels = 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout2d(0.3)
        self.pool = nn.MaxPool2d(kernel_size = 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)

        return x


In [ ]:
class FlexibleCrackSeg(nn.Module):
    def __init__(self, num_blocks, num_filters):
        super(FlexibleCrackSeg, self).__init__()

        self.num_blocks = num_blocks
        self.num_filters = num_filters

        self.conv_blocks = nn.ModuleList([])

        for i in range(num_blocks):
            self.conv_blocks.append(ConvBlock(num_filters * (2 ** i)))

        self.flatten = nn.Flatten()
        self.fc1 = None

        def forward(self, x):
            for block in self.conv_blocks:
                x = block(x)